In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_PATH = '/content/drive/MyDrive/CSCI 5527 Group Project/'
BEAUTY_PATH  = os.path.join(PROJECT_PATH, 'beauty_rec')
print('Project :', os.listdir(PROJECT_PATH))
print('Beauty  :', os.listdir(BEAUTY_PATH))

Project : ['Electronics.train.csv.gz', 'Electronics.valid.csv.gz', 'Electronics.test.csv.gz', 'Draft - idea.pdf', 'Yelp_Dataset.ipynb', 'Draft_notebook_dataset_exploration (1).ipynb', 'Lightning Talk.gslides', 'Progress Report Draft.gdoc', 'Lightning Talk Script.gdoc', 'Draft_notebook_dataset_exploration_(1).ipynb', 'Papers', 'fusion.ipynb', 'evaluation.ipynb', 'Final Report Draft.gdoc', 'Untitled (8).ipynb', 'embeddings.ipynb', 'beauty_rec', 'Beauty Amazon.ipynb']
Beauty  : ['amazon_beauty_2023_full.csv', 'amazon_beauty_cleaned.csv', 'amazon_beauty_2M.csv', 'text_embeddings.npy', 'image_embeddings.npy', 'user_map.csv', 'interactions.csv', 'item_map.csv']


In [ ]:
import os, numpy as np, pandas as pd
df = pd.read_csv(os.path.join(BEAUTY_PATH, 'interactions.csv'))

item_map   = pd.read_csv(os.path.join(BEAUTY_PATH, 'item_map.csv'))
ITEM_COUNT = int(item_map['item_idx'].max())
MAX_LEN    = 50

print(f'Full dataset: {df["user_idx"].nunique():,} users, {ITEM_COUNT:,} items')
print(df['split'].value_counts().to_string())
SUBSAMPLE_FRAC = 0.20
SEED           = 42
rng            = np.random.default_rng(SEED)

train_counts   = df[df['split'] == 'train']['item_idx'].value_counts().to_dict()

def cold_tier(item_idx):
    c = train_counts.get(item_idx, 0)
    return 'cold' if c == 0 else ('few_shot' if c < 5 else 'warm')
test_df        = df[df['split'] == 'test'].copy()
test_df['tier'] = test_df['item_idx'].map(cold_tier)

sampled_users = []
for tier, group in test_df.groupby('tier'):
    n      = max(1, int(len(group) * SUBSAMPLE_FRAC))
    chosen = rng.choice(group['user_idx'].values, size=n, replace=False)
    sampled_users.extend(chosen.tolist())
    print(f'  {tier:<10}: sampled {n:,} / {len(group):,} test users')
sampled_users = set(sampled_users)
df_sub        = df[df['user_idx'].isin(sampled_users)].copy()

print(f'\nSubsample: {df_sub["user_idx"].nunique():,} users')
print(df_sub['split'].value_counts().to_string())

Full dataset: 631,915 users, 112,553 items
split
train    677545
val        8151
test       8151
  cold      : sampled 209 / 1,045 test users
  few_shot  : sampled 386 / 1,930 test users
  warm      : sampled 1,035 / 5,176 test users

Subsample: 1,630 users
split
train    3825
val      1630
test     1630


In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class TiSASRecDataset(Dataset):
    def __init__(self, df, mode='train', max_len=50, num_neg=999, seed=42):
        assert mode in ('train', 'val', 'test')
        self.max_len  = max_len
        self.mode     = mode
        self.num_neg  = num_neg
        self.rng      = np.random.default_rng(seed)
        self.all_items = np.arange(1, df['item_idx'].max() + 1)

        train_df    = df[df['split'] == 'train']
        train_seqs  = train_df.groupby('user_idx')['item_idx'].apply(list)
        train_times = train_df.groupby('user_idx')['timestamp'].apply(list)
        user_seen   = df.groupby('user_idx')['item_idx'].apply(set).to_dict()

        self.records = []
        if mode == 'train':
            for uid, seq in train_seqs.items():
                if len(seq) >= 2:
                    self.records.append({
                        'history': seq[:-1], 'times': train_times[uid][:-1],
                        'target': seq[-1],   'seen':  user_seen[uid],
                    })
        else:
            tgt_df    = df[df['split'] == mode]
            tgt_items = tgt_df.groupby('user_idx')['item_idx'].first().to_dict()
            for uid, target in tgt_items.items():
                seq = train_seqs.get(uid)
                if seq is None or len(seq) < 1:
                    continue
                self.records.append({
                    'history': list(seq), 'times': list(train_times[uid]),
                    'target': target,     'seen':  user_seen[uid],
                })

        print(f'[Dataset:{mode}] {len(self.records):,} users')

    def _sample_negatives(self, seen, n):
        negs = []
        while len(negs) < n:
            cands = self.rng.choice(self.all_items, size=n * 2, replace=False)
            negs += [c for c in cands.tolist() if c not in seen]
        return negs[:n]

    def __len__(self):  return len(self.records)

    def __getitem__(self, idx):
        rec  = self.records[idx]
        hist = rec['history'][-(self.max_len):]
        ts   = rec['times']  [-(self.max_len):]

        raw_iv    = [ts[i+1] - ts[i] for i in range(len(ts)-1)] + [0.0]
        intervals = [float(np.log1p(max(dt, 0))) for dt in raw_iv]

        L     = len(hist)
        s_arr = np.zeros(self.max_len, dtype=np.int64)
        i_arr = np.zeros(self.max_len, dtype=np.float32)
        s_arr[-L:] = hist
        i_arr[-L:] = intervals

        if self.mode == 'train':
            neg = self._sample_negatives(rec['seen'], 1)[0]
            return (torch.from_numpy(s_arr), torch.from_numpy(i_arr),
                    torch.tensor(rec['target'], dtype=torch.long),
                    torch.tensor(neg, dtype=torch.long))
        else:
            negs = self._sample_negatives(rec['seen'], self.num_neg)
            cands = torch.tensor([rec['target']] + negs, dtype=torch.long)
            return torch.from_numpy(s_arr), torch.from_numpy(i_arr), cands

def evaluate(model, df_eval, max_len, batch_size=256,
             num_neg=999, ks=(5, 10, 20), label=''):
    """Run sampled evaluation and return a results dict."""
    train_counts = df_eval[df_eval['split']=='train']['item_idx'].value_counts().to_dict()

    dataset = TiSASRecDataset(df_eval, mode='test', max_len=max_len, num_neg=num_neg)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                         num_workers=2, pin_memory=True)

    tiers   = ('cold', 'few_shot', 'warm')
    results = {t: {f'{m}@{k}': [] for k in ks for m in ('recall','ndcg')}
               for t in tiers}

    model.eval()
    with torch.no_grad():
        for seq, inter, candidates in loader:
            scores  = model(seq.cuda(), inter.cuda(), candidates.cuda()).cpu().numpy()
            for i in range(len(candidates)):
                target = candidates[i, 0].item()
                rank   = int((scores[i, 1:] >= scores[i, 0]).sum()) + 1
                count  = train_counts.get(target, 0)
                tier   = 'cold' if count==0 else ('few_shot' if count<5 else 'warm')
                for k in ks:
                    results[tier][f'recall@{k}'].append(1 if rank<=k else 0)
                    results[tier][f'ndcg@{k}'].append(
                        1/np.log2(rank+1) if rank<=k else 0.0)

    k_hdr  = ''.join(f" | {'R@'+str(k):<7} {'N@'+str(k):<7}" for k in ks)
    header = f"{'Tier':<12}{k_hdr} | Samples"
    sep    = '─' * len(header)
    tag    = f'  [{label}]' if label else ''
    print(f'\n{sep}{tag}')
    print(header); print(sep)

    summary = {}
    for tier in tiers:
        n   = len(results[tier][f'recall@{ks[0]}'])
        row = f'{tier:<12}'
        for k in ks:
            r = np.mean(results[tier][f'recall@{k}']) if n else 0.0
            d = np.mean(results[tier][f'ndcg@{k}'])   if n else 0.0
            row += f' | {r:<7.4f} {d:<7.4f}'
            summary[f'{tier}_recall@{k}'] = round(r, 4)
            summary[f'{tier}_ndcg@{k}']   = round(d, 4)
        print(f'{row} | {n}')
    print(sep)
    row = f"{'overall':<12}"
    all_n = 0
    for k in ks:
        flat_r = [v for t in tiers for v in results[t][f'recall@{k}']]
        flat_d = [v for t in tiers for v in results[t][f'ndcg@{k}']]
        r = np.mean(flat_r) if flat_r else 0.0
        d = np.mean(flat_d) if flat_d else 0.0
        row += f' | {r:<7.4f} {d:<7.4f}'
        summary[f'overall_recall@{k}'] = round(r, 4)
        summary[f'overall_ndcg@{k}']   = round(d, 4)
        all_n = len(flat_r)
    print(f'{row} | {all_n}')
    print(sep)
    return summary

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np

class MultimodalFusion(nn.Module):
    def __init__(self, text_dim=384, image_dim=2048, inter_dim=128, shared_dim=128):
        super().__init__()
        self.text_proj  = nn.Linear(text_dim,  shared_dim)
        self.image_proj = nn.Linear(image_dim, shared_dim)
        self.inter_proj = nn.Linear(inter_dim, shared_dim)
        self.attn_i2t   = nn.MultiheadAttention(shared_dim, num_heads=4, batch_first=True)
        self.attn_t2i   = nn.MultiheadAttention(shared_dim, num_heads=4, batch_first=True)
        self.gate = nn.Sequential(nn.Linear(shared_dim*2, shared_dim), nn.Sigmoid())
        self.out  = nn.Linear(shared_dim, shared_dim)

    def forward(self, text_emb, image_emb, interaction_emb):
        t = self.text_proj(text_emb).unsqueeze(1)
        v = self.image_proj(image_emb).unsqueeze(1)
        r = self.inter_proj(interaction_emb)
        v2t = self.attn_i2t(v, t, t)[0].squeeze(1)
        t2v = self.attn_t2i(t, v, v)[0].squeeze(1)
        g   = self.gate(torch.cat([v2t, t2v], dim=-1))
        return self.out(g * v2t + (1-g) * t2v + r)

class SequenceEncoder(nn.Module):
    def __init__(self, item_count, shared_dim, max_len):
        super().__init__()
        self.item_emb = nn.Embedding(item_count+1, shared_dim, padding_idx=0)
        self.pos_emb  = nn.Embedding(max_len, shared_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(1, shared_dim), nn.ReLU(), nn.Linear(shared_dim, shared_dim))
        enc = nn.TransformerEncoderLayer(
            d_model=shared_dim, nhead=8, batch_first=True,
            dim_feedforward=shared_dim*4, dropout=0.2, norm_first=False)
        self.transformer = nn.TransformerEncoder(enc, num_layers=2)
        self.norm        = nn.LayerNorm(shared_dim)

    def encode(self, seq, intervals, item_repr):
        B, S     = seq.shape
        time_enc = self.time_mlp(intervals.unsqueeze(-1))
        pos_enc  = self.pos_emb(torch.arange(S, device=seq.device))
        x        = self.norm(item_repr + time_enc + pos_enc)
        mask     = torch.triu(torch.ones(S, S, device=seq.device), 1).bool()
        out      = self.transformer(x, mask=mask)
        return out[:, -1, :]    # [B, D]

    def score_candidates(self, user_repr, candidates):
        cand_emb = self.item_emb(candidates)
        return torch.bmm(cand_emb, user_repr.unsqueeze(-1)).squeeze(-1)

class IDOnlyModel(SequenceEncoder):
    """Uses only item ID embeddings — no modality features."""
    def forward(self, seq, intervals, candidates):
        ids       = self.item_emb(seq)                  # [B,S,D]
        user_repr = self.encode(seq, intervals, ids)
        return self.score_candidates(user_repr, candidates)

class TextOnlyModel(nn.Module):
    """Fuses text embeddings with item ID; image branch zeroed out."""
    def __init__(self, item_count, shared_dim, txt_path, max_len=50):
        super().__init__()
        txt_np  = np.load(txt_path).astype(np.float32)
        txt_np /= np.linalg.norm(txt_np, axis=1, keepdims=True).clip(min=1e-8)
        self.register_buffer('txt_feats', torch.from_numpy(txt_np))

        self.text_proj = nn.Linear(384, shared_dim)
        self.backbone  = SequenceEncoder(item_count, shared_dim, max_len)

    def forward(self, seq, intervals, candidates):
        txt  = F.pad(self.txt_feats, (0,0,1,0))[seq]
        ids  = self.backbone.item_emb(seq)
        # Simple add: projected text + ID
        item_repr = self.text_proj(txt.reshape(-1, 384)).reshape(*ids.shape) + ids
        user_repr = self.backbone.encode(seq, intervals, item_repr)
        return self.backbone.score_candidates(user_repr, candidates)

class ImageOnlyModel(nn.Module):
    """Fuses image embeddings with item ID; text branch zeroed out."""
    def __init__(self, item_count, shared_dim, img_path, max_len=50):
        super().__init__()
        img_np  = np.load(img_path).astype(np.float32)
        img_np /= np.linalg.norm(img_np, axis=1, keepdims=True).clip(min=1e-8)
        self.register_buffer('img_feats', torch.from_numpy(img_np))

        self.image_proj = nn.Linear(2048, shared_dim)
        self.backbone   = SequenceEncoder(item_count, shared_dim, max_len)

    def forward(self, seq, intervals, candidates):
        img  = F.pad(self.img_feats, (0,0,1,0))[seq]
        ids  = self.backbone.item_emb(seq)
        item_repr = self.image_proj(img.reshape(-1,2048)).reshape(*ids.shape) + ids
        user_repr = self.backbone.encode(seq, intervals, item_repr)
        return self.backbone.score_candidates(user_repr, candidates)

class FullModel(nn.Module):
    """Full multimodal fusion: text + image + ID with cross-attention gating."""
    def __init__(self, item_count, shared_dim, img_path, txt_path, max_len=50):
        super().__init__()
        img_np  = np.load(img_path).astype(np.float32)
        txt_np  = np.load(txt_path).astype(np.float32)
        img_np /= np.linalg.norm(img_np, axis=1, keepdims=True).clip(min=1e-8)
        txt_np /= np.linalg.norm(txt_np, axis=1, keepdims=True).clip(min=1e-8)
        self.register_buffer('img_feats', torch.from_numpy(img_np))
        self.register_buffer('txt_feats', torch.from_numpy(txt_np))

        self.fusion   = MultimodalFusion(shared_dim=shared_dim)
        self.backbone = SequenceEncoder(item_count, shared_dim, max_len)

    def forward(self, seq, intervals, candidates):
        B, S = seq.shape
        img  = F.pad(self.img_feats, (0,0,1,0))[seq]
        txt  = F.pad(self.txt_feats, (0,0,1,0))[seq]
        ids  = self.backbone.item_emb(seq)

        item_repr = self.fusion(
            txt.reshape(-1, txt.shape[-1]),
            img.reshape(-1, img.shape[-1]),
            ids.reshape(-1, ids.shape[-1]),
        ).reshape(B, S, -1)

        user_repr = self.backbone.encode(seq, intervals, item_repr)
        return self.backbone.score_candidates(user_repr, candidates)
print('✔  All 4 model variants defined: IDOnly, TextOnly, ImageOnly, FullModel')

✔  All 4 model variants defined: IDOnly, TextOnly, ImageOnly, FullModel


In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader

NUM_EPOCHS = 30
BATCH_SIZE = 256
SHARED_DIM = 128
IMG_PATH   = os.path.join(BEAUTY_PATH, 'image_embeddings.npy')
TXT_PATH   = os.path.join(BEAUTY_PATH, 'text_embeddings.npy')

def bpr_loss(pos, neg):
    return -torch.log(torch.sigmoid(pos - neg) + 1e-8).mean()

def train_and_eval(model, df_sub, label):
    """Train a model on df_sub and evaluate it. Returns metrics dict."""
    print(f'\n{"="*60}')
    print(f'  Variant: {label}')
    print(f'{"="*60}')

    train_ds = TiSASRecDataset(df_sub, mode='train', max_len=MAX_LEN)
    loader   = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)

    opt  = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS, eta_min=1e-5)
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        total, n = 0.0, 0
        for seq, inter, pos, neg in loader:
            seq, inter, pos, neg = seq.cuda(), inter.cuda(), pos.cuda(), neg.cuda()
            opt.zero_grad()
            scores = model(seq, inter, torch.stack([pos, neg], dim=1))
            loss   = bpr_loss(scores[:, 0], scores[:, 1])
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            total += loss.item(); n += 1
        sched.step()
        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:02d}/{NUM_EPOCHS} | BPR Loss: {total/max(n,1):.4f}')
    metrics = evaluate(model, df_sub, max_len=MAX_LEN, label=label)
    return metrics

all_results = {}
model = IDOnlyModel(ITEM_COUNT, SHARED_DIM, MAX_LEN).cuda()
all_results['ID only'] = train_and_eval(model, df_sub, 'ID only')

model = TextOnlyModel(ITEM_COUNT, SHARED_DIM, TXT_PATH, MAX_LEN).cuda()
all_results['Text only'] = train_and_eval(model, df_sub, 'Text only')

model = ImageOnlyModel(ITEM_COUNT, SHARED_DIM, IMG_PATH, MAX_LEN).cuda()
all_results['Image only'] = train_and_eval(model, df_sub, 'Image only')
model = FullModel(ITEM_COUNT, SHARED_DIM, IMG_PATH, TXT_PATH, MAX_LEN).cuda()
all_results['Text + Image'] = train_and_eval(model, df_sub, 'Text + Image (Full)')


  Variant: ID only
[Dataset:train] 560 users
  Epoch 01/30 | BPR Loss: 5.6488
  Epoch 05/30 | BPR Loss: 2.6301
  Epoch 10/30 | BPR Loss: 1.8619
  Epoch 15/30 | BPR Loss: 1.3216
  Epoch 20/30 | BPR Loss: 0.9165
  Epoch 25/30 | BPR Loss: 0.6741
  Epoch 30/30 | BPR Loss: 0.5442
[Dataset:test] 1,630 users

────────────────────────────────────────────────────────────────────────────  [ID only]
Tier         | R@5     N@5     | R@10    N@10    | R@20    N@20    | Samples
────────────────────────────────────────────────────────────────────────────
cold         | 0.0066  0.0040  | 0.0148  0.0066  | 0.0251  0.0092  | 1354
few_shot     | 0.0073  0.0052  | 0.0073  0.0052  | 0.0182  0.0079  | 274
warm         | 0.0000  0.0000  | 0.0000  0.0000  | 0.0000  0.0000  | 2
────────────────────────────────────────────────────────────────────────────
overall      | 0.0067  0.0042  | 0.0135  0.0064  | 0.0239  0.0090  | 1630
────────────────────────────────────────────────────────────────────────────

  Vari

In [ ]:
import pandas as pd

KS      = (5, 10, 20)
METRICS = [f'{m}@{k}' for k in KS for m in ('recall','ndcg')]

rows = []
for variant, res in all_results.items():
    row = {'Variant': variant}
    for m in METRICS:
        row[f'warm_{m}']     = res.get(f'warm_{m}', 0.0)
        row[f'few_shot_{m}'] = res.get(f'few_shot_{m}', 0.0)
        row[f'cold_{m}']     = res.get(f'cold_{m}', 0.0)
        row[f'overall_{m}']  = res.get(f'overall_{m}', 0.0)
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Variant')

print('\n' + '='*70)
print('  ABLATION STUDY SUMMARY  (20% stratified subsample)')
print('='*70)
print(f"{'Variant':<20} | {'R@10':>7} {'N@10':>7} | {'warm R@10':>10} {'warm N@10':>10} | {'cold R@10':>10}")
print('-'*70)
for variant, res in all_results.items():
    print(
        f"{variant:<20} | "
        f"{res.get('overall_recall@10',0):.4f}  {res.get('overall_ndcg@10',0):.4f} | "
        f"{res.get('warm_recall@10',0):.4f}      {res.get('warm_ndcg@10',0):.4f}     | "
        f"{res.get('cold_recall@10',0):.4f}"
    )
print('='*70)
out_path = os.path.join(BEAUTY_PATH, 'ablation_results.csv')
summary_df.to_csv(out_path)
print(f'\n✔  Full results saved to {out_path}')


  ABLATION STUDY SUMMARY  (20% stratified subsample)
Variant              |    R@10    N@10 |  warm R@10  warm N@10 |  cold R@10
----------------------------------------------------------------------
ID only              | 0.0135  0.0064 | 0.0000      0.0000     | 0.0148
Text only            | 0.0074  0.0032 | 0.0000      0.0000     | 0.0089
Image only           | 0.0080  0.0042 | 0.0000      0.0000     | 0.0059
Text + Image         | 0.0098  0.0042 | 0.0000      0.0000     | 0.0118

✔  Full results saved to /content/drive/MyDrive/CSCI 5527 Group Project/beauty_rec/ablation_results.csv
